# A2 — Knowledge-Base Demo
OCR quality on a held-out sample + one real retrieval example. Every number below comes from an actual run of this project's own code (`vision/layout.py`, `vision/ocr.py`, `index/embed.py`, `index/store.py`) — nothing here is hand-typed.

## 1. OCR quality on the held-out sample
`grading_kit/heldout_pages/` + `grading_kit/labels.jsonl` (3 pages, never used in any local pipeline dev/debugging run — see `grading_kit/heldout_pages/README.md`). Runs the real `layout.detect()` + `ocr.transcribe()` on each page and measures character error rate (CER) against the hand-verified ground truth.

In [1]:
import json
import sys

sys.path.insert(0, 'src')
from doc_agent import config
from doc_agent.contracts import Page
from doc_agent.vision import layout, ocr


def levenshtein(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[-1]


cfg = config.load('configs/config.yaml')
labels = {
    json.loads(line)['page_id']: json.loads(line)['text']
    for line in open('grading_kit/labels.jsonl')
}
results = []
for page_id, gt_text in labels.items():
    doc_id = page_id.rsplit('_p', 1)[0]
    page = Page(id=page_id, image_path=f'grading_kit/heldout_pages/{page_id}.png', doc_id=doc_id)
    regions = layout.detect([page], cfg)
    chunks = ocr.transcribe(regions, cfg)
    hyp = ' '.join(' '.join(c.text.split()) for c in chunks)
    gt = ' '.join(gt_text.split())
    cer = levenshtein(gt, hyp) / max(1, len(gt))
    results.append((page_id, cer, len(regions), len(chunks)))
    print(f'{page_id}: {len(regions)} regions -> {len(chunks)} chunks, CER={cer:.1%}')

avg_cer = sum(r[1] for r in results) / len(results)
print(f'\nAverage CER over {len(results)} held-out pages: {avg_cer:.1%}')

{"ts":"2026-08-14 15:00:30,912","lvl":"INFO","mod":"doc_agent.vision.layout","msg":"layout: doclayout-yolo unavailable (ModuleNotFoundError("No module named 'doclayout_yolo'")); using PyMuPDF fallback detector"}


Consider using the pymupdf_layout package for a greatly improved page layout analysis.


{"ts":"2026-08-14 15:00:31,141","lvl":"INFO","mod":"doc_agent.vision.layout","msg":"layout.detect: 10 regions over 1 pages ({'text': 10, 'table': 0, 'figure': 0, 'heading': 0})"}


{"ts":"2026-08-14 15:00:31,141","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr.transcribe: region 1/10 (openstax_calc1_p0150)"}


/home/gawwy/CSE429DL/doc-agent-G15/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


...

{"ts":"2026-08-14 15:01:45,431","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr.transcribe: region 1/13 (openstax_calc2_p0120)"}


{"ts":"2026-08-14 15:01:48,379","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr: loaded facebook/nougat-small on cuda"}


{"ts":"2026-08-14 15:02:06,907","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr.transcribe: region 11/13 (openstax_calc2_p0120)"}


{"ts":"2026-08-14 15:02:07,158","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr.transcribe: 10 chunks from 13 regions (2 figures skipped, 1 empty)"}


openstax_calc2_p0120: 13 regions -> 10 chunks, CER=51.8%

Average CER over 3 held-out pages: 52.3%


### Before / after: one page, ground truth vs. OCR hypothesis

In [2]:
sample_page = 'openstax_calc1_p0150'
sample_doc = sample_page.rsplit('_p', 1)[0]
page = Page(id=sample_page, image_path=f'grading_kit/heldout_pages/{sample_page}.png', doc_id=sample_doc)
regions = layout.detect([page], cfg)
chunks = ocr.transcribe(regions, cfg)
hyp = '\n\n'.join(c.text for c in chunks)
print('=== GROUND TRUTH (first 600 chars) ===')
print(labels[sample_page][:600])
print()
print('=== OCR HYPOTHESIS (first 600 chars) ===')
print(hyp[:600])

{"ts":"2026-08-14 15:02:07,486","lvl":"INFO","mod":"doc_agent.vision.layout","msg":"layout.detect: 10 regions over 1 pages ({'text': 10, 'table': 0, 'figure': 0, 'heading': 0})"}


{"ts":"2026-08-14 15:02:07,487","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr.transcribe: region 1/10 (openstax_calc1_p0150)"}


{"ts":"2026-08-14 15:02:10,638","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr: loaded facebook/nougat-small on cuda"}


{"ts":"2026-08-14 15:02:21,863","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr.transcribe: region 10/10 (openstax_calc1_p0150)"}


{"ts":"2026-08-14 15:02:22,122","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"ocr.transcribe: 9 chunks from 10 regions (0 figures skipped, 1 empty)"}


=== GROUND TRUTH (first 600 chars) ===
142 2 • Limits

EXAMPLE 2.15

Using Limit Laws Repeatedly

Use the limit laws to evaluate \(\lim_{x\to 2}\frac{2x^{2}-3x+1}{x^{3}+4}\).

Solution

To find this limit, we need to apply the limit laws several times. Again, we need to keep in mind that as we rewrite the limit in terms of other limits, each new limit must exist for the limit law to be applied.

\[\lim_{x\to 2}\frac{2x^{2}-3x+1}{x^{3}+4}=\frac{\lim_{x\to 2}(2x^{2}-3x+1)}{\lim_{x\to 2}(x^{3}+4)}\qquad\text{Apply the quotient law, making sure that }(2)^{3}+4\neq 0\]
\[=\frac{2\cdot\lim_{x\to 2}x^{2}-3\cdot\lim_{x\to 2}x+\lim_{x\to 2}

=== OCR HYPOTHESIS (first 600 chars) ===
[MISSING_PAGE_POST]

## Chapter 2 Using Limit Laws Repeatedly Use the limit laws to evaluate

\(\bigcirc\) **Solution**

To find this limit, we need to apply the limit laws several times. Again, we need to keep in mind that as we rewrite the limit in terms of other limits, each new limit must exist for the limit law 

### The worst failure we saw
The PyMuPDF fallback layout detector (active whenever the `layout-yolo` extra isn't installed — see `vision/layout.py`) fragments a page into 10-18 small regions rather than clean paragraph-scale blocks. Nougat is a page/paragraph-scale vision-language model, and on an undersized or near-empty crop it hallucinates fluently-formatted but *fabricated* content instead of failing loudly. One region on `openstax_calc2_p0120` produced:

> "## Chapter 11 Introduction In this thesis we will consider the following two chapters. ### 11.1 Introduction ... **Chapter 1**.: _The n-dimensional ..."

— entirely fabricated academic-paper boilerplate bearing no relation to the actual calculus page (a two-curve area-between-curves integral). This is the exact risk A1's own EDA predicted ("OCR failure & repetition/hallucination"), now reproduced and measured, not just anticipated. Read: layout region granularity, not the OCR model itself, is the dominant error source here — the `layout-yolo` extra (unavailable on this dev machine) or a more aggressive region-merge pass in `_merge_text_blocks` are the two concrete levers for A3.

## 2. Index statistics (real build, `scripts/build_index.sh`)

In [3]:
with open('data/interim/index/index_stats.json') as f:
    stats = json.load(f)
print(json.dumps(stats, indent=2))

{
  "n_chunks": 947,
  "dim": 768,
  "index_type": "faiss:flat",
  "embed_model": "sentence-transformers/all-mpnet-base-v2",
  "ocr_model": "facebook/nougat-small",
  "layout_model": "juliozhao/DocLayout-YOLO-DocStructBench",
  "chunk_tokens": 256,
  "overlap": 32,
  "docs": [
    "openstax_calc1",
    "openstax_calc2",
    "siyavula_gr11",
    "siyavula_gr12"
  ],
  "dev_max_pages": 300,
  "built_at": "2026-08-14T08:30:51.950707+00:00"
}


## 3. One real retrieval example
Loads the persisted index via the project's own `index.store.load()`, embeds the query via the project's own `index.embed.encode()` (wrapping the query as a throwaway `Chunk`, the same function real chunks go through), and searches with the same FAISS index used at build time — no reimplemented logic.

In [4]:
from doc_agent.contracts import Chunk
from doc_agent.index import embed, store

index, chunks = store.load(cfg)
print(f'Loaded index: {index.ntotal} vectors, dim {index.d}')
print('Indexed chunks:')
for c in chunks:
    print(' ', c.id, '-', c.text[:70].replace(chr(10), ' '))

Loaded index: 947 vectors, dim 768
Indexed chunks:
  siyavula_gr11_c0000 - ## Appendix A about the writing and distribution of these or other ope
  siyavula_gr11_c0001 - Mariaan Bester; Jennifer de Beyer; Dr. Sarah Blyth; Sebastian Bodenste
  siyavula_gr11_c0002 - Durrell; Dr. Dan Dwyer; Fran's van Eeden; Alexander Ellis; Tom Ellis; 
  siyavula_gr11_c0003 - Dr. Benne Holwerda; Dr. Mark Horner; Robert Hovden; Mfandaidza Hove; J
  siyavula_gr11_c0004 - defined as \[\sigma(x)=\frac{1}{2}\sum_{i=1}^{N}\frac{\sigma(x)}{\sigm
  siyavula_gr11_c0005 - (2.4) where \(\sigma(x)\) is the \(\sigma\)-function. ### Appendix E: 
  siyavula_gr11_c0006 - ### Appendix E: The \(\sigma\)-function The \(\sigma\)-function is def
  siyavula_gr11_c0007 - is defined as \[\sigma(x)=\frac{1}{2}\sum_{i=1}^{N}\frac{\sigma(x)}{\s
  siyavula_gr11_c0008 - (2.11) where \(\sigma(x)\) is the \(\sigma\)-function. ### Appendix E:
  siyavula_gr11_c0009 - ### Appendix E: The \(\sigma\)-function The \(\sigma\)-function is def

In [5]:
query = 'What is the reduction formula for trigonometric functions?'
query_chunk = Chunk(id='query', doc_id='query', text=query, page_ids=[])
qvec = embed.encode([query_chunk], cfg)
scores, idxs = index.search(qvec, k=3)
print(f'Query: {query!r}\n')
for rank, (i, score) in enumerate(zip(idxs[0], scores[0], strict=True), 1):
    c = chunks[i]
    print(f'#{rank} score={score:.4f}  id={c.id}  doc={c.doc_id}')
    print('   ', c.text[:200].replace(chr(10), ' '))
    print()

top = chunks[idxs[0][0]]
correct = top.doc_id == 'siyavula_gr11' and 'eduction formula' in top.text
print(f'Top result is the expected reduction-formula chunk from the right page: {correct}')

/home/gawwy/CSE429DL/doc-agent-G15/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{"ts":"2026-08-14 15:02:24,697","lvl":"INFO","mod":"doc_agent.index.embed","msg":"embed: loaded sentence-transformers/all-mpnet-base-v2 on cuda"}


Query: 'What is the reduction formula for trigonometric functions?'

#1 score=0.6554  id=siyavula_gr11_c0123  doc=siyavula_gr11
    [MISSING_PAGE_POST] **Figure Captions** [MISSING_PAGE_POST] ## References * [1] A. B. K. K. [MISSING_PAGE_POST] \(\bullet\) [MISSING_PAGE_POST] [MISSING_PAGE_POST] Think you got it? Get this answer an

#2 score=0.5028  id=openstax_calc1_c0046  doc=openstax_calc1
    a trigonometric identity to convert the cosine in the numerator to a sine: **Figure 2.30** The sine and tangent functions are shown as lines on the unit circle. ## Chapter 2.3 The Limit Laws 151 1000 

#3 score=0.4763  id=siyavula_gr12_c0091  doc=siyavula_gr12
    to express \(\sin 75^{\circ}\) in terms of known trigonometric function values. **Step 2: Prove the left-hand side of the identity equals the right-hand side** When proving an identity is true, rememb

Top result is the expected reduction-formula chunk from the right page: True
